# L8 · Notebook 03 — 最小化 DQN：CartPole capstone

**对应 PPT**：`L8-VFA.tex` §4 DQN 算法 / 经验回放 / 目标网络

## 教学目标

用最少的代码实现 Mnih et al. 2015 的 DQN 三件套：

1. **Q 网络**：2 层 MLP，输入 4 维状态，输出 2 个动作的 $Q(s,\cdot;w)$
2. **经验回放**：环形 buffer，打破时间相关性 → 恢复 RM 假设 (C2) i.i.d.
3. **目标网络** $w^-$：每 $C$ 步硬同步，把 bootstrap 部分『冻结』成回归任务
4. **ε-贪婪**探索 + 线性退火

在 Gymnasium `CartPole-v1` 上训练 ~300 episode，return 应从 ~20 升到 200+（满分 500）。

In [ ]:
import random
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

## 1. Q 网络（2 层 MLP）

In [ ]:
class QNet(nn.Module):
    def __init__(self, n_state=4, n_action=2, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_state, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.ReLU(),
            nn.Linear(hidden, n_action),
        )
    def forward(self, x):
        return self.net(x)

qnet  = QNet().to(device)
qtgt  = QNet().to(device); qtgt.load_state_dict(qnet.state_dict())
opt   = optim.Adam(qnet.parameters(), lr=1e-3)
print(qnet)

## 2. 经验回放（环形 buffer）

In [ ]:
class ReplayBuffer:
    def __init__(self, cap=10000):
        self.buf = deque(maxlen=cap)
    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))
    def sample(self, batch):
        idx = random.sample(range(len(self.buf)), batch)
        s, a, r, s2, done = zip(*[self.buf[i] for i in idx])
        return (torch.tensor(np.array(s),  dtype=torch.float32, device=device),
                torch.tensor(a,            dtype=torch.long,    device=device),
                torch.tensor(r,            dtype=torch.float32, device=device),
                torch.tensor(np.array(s2), dtype=torch.float32, device=device),
                torch.tensor(done,         dtype=torch.float32, device=device))
    def __len__(self): return len(self.buf)

buf = ReplayBuffer(cap=10000)

## 3. 训练循环

三件套：
- **ε-greedy** 行为策略（off-policy）
- **replay buffer** 取小批量 → 打散时间相关性
- **target net** $w^-$ 每 `C=100` 步硬同步

损失：$L(w) = \mathbb E\bigl[(r + \gamma\,\max_{a'} Q(s',a';w^-) - Q(s,a;w))^2\bigr]$

In [ ]:
def select_action(state, eps):
    if random.random() < eps:
        return random.randrange(2)
    with torch.no_grad():
        s = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        return int(qnet(s).argmax(dim=1).item())

def train_step(batch_size=64, gamma=0.99):
    if len(buf) < batch_size:
        return None
    s, a, r, s2, done = buf.sample(batch_size)
    q  = qnet(s).gather(1, a.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        q_tgt = qtgt(s2).max(dim=1)[0]
        target = r + gamma * (1 - done) * q_tgt
    loss = nn.functional.smooth_l1_loss(q, target)
    opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(qnet.parameters(), 10.0)
    opt.step()
    return loss.item()

In [ ]:
env = gym.make('CartPole-v1')
N_EPISODES   = 300
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.05, 200  # 线性退火 episode 数
TARGET_SYNC  = 100      # C
BATCH_SIZE   = 64
GAMMA        = 0.99

returns = []
global_step = 0
for ep in range(N_EPISODES):
    s, _ = env.reset(seed=SEED + ep)
    eps = max(EPS_END, EPS_START - (EPS_START - EPS_END) * ep / EPS_DECAY)
    ep_ret = 0.0
    while True:
        a = select_action(s, eps)
        s2, r, term, trunc, _ = env.step(a)
        done = term or trunc
        buf.push(s, a, r, s2, float(term))   # 截断 ≠ terminal
        s = s2
        ep_ret += r
        global_step += 1
        train_step(BATCH_SIZE, GAMMA)
        if global_step % TARGET_SYNC == 0:
            qtgt.load_state_dict(qnet.state_dict())
        if done:
            break
    returns.append(ep_ret)
    if (ep + 1) % 25 == 0:
        recent = np.mean(returns[-25:])
        print(f'ep {ep+1:3d} | eps={eps:.2f} | recent-25 mean return = {recent:6.1f}')

env.close()

## 4. 学习曲线

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(returns, alpha=0.3, label='per-episode return')
if len(returns) >= 25:
    smooth = np.convolve(returns, np.ones(25)/25, mode='valid')
    ax.plot(np.arange(24, len(returns)), smooth, 'C1', linewidth=2, label='25-ep moving avg')
ax.axhline(195, color='red', linestyle='--', alpha=0.5, label='solved threshold (195)')
ax.set_xlabel('episode'); ax.set_ylabel('return')
ax.set_title('DQN on CartPole-v1 (minimal: replay + target net)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/dqn_cartpole_returns.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. 课堂诊断小结

| 模块 | 解决什么问题 | 对应 RM 假设 |
|---|---|---|
| Q 网络（FA） | 状态太大无法表格 | — |
| 经验回放 | 时间相关 → 打散为 i.i.d. | (C2) 噪声零均值 |
| 目标网络 | bootstrap 不动 → 退化为回归 | (C3) 稳定不动点 |
| ε-greedy | 探索 | — |

**实测**：在 CPU 上 300 episode（~3 分钟）能稳超 100 分，~500 episode 通常解决 CartPole。

## 思考题

1. 把 `TARGET_SYNC = 100` 改成 `1`（每步同步），训练会怎么样？为什么？
2. 把 `ReplayBuffer` 容量改成 100 + 不打乱，相当于 online 训练 → 复现 Baird 式不稳？
3. 把 `nn.SmoothL1Loss` 换成 `MSELoss`，对 outlier reward 鲁棒性变化？
4. **Double DQN**（§5）只需改 1 行：`q_tgt = qtgt(s2).gather(1, qnet(s2).argmax(dim=1, keepdim=True)).squeeze(1)`，对比 max bias。